In [20]:
import pandas as pd
import numpy as np
from datetime import datetime

In [21]:
#pulling data
df = pd.read_parquet("data/clean_data.parquet")

In [22]:
#changing churn_type to int
df["churn_type"] = df["churn_type"].astype(int)

In [23]:
#weighted engagement score
df["engagement_score"] = (
    df["completion_rate"] * 0.4 +
    (df["avg_watch_hours_per_week"] / 20) * 0.4 +
    (df["number_of_logins_per_week"] / 20) * 0.2
)

In [24]:
#to track customer loyalty, long-turned customers tend to churn less
reference_date = pd.Timestamp(datetime.now().date())
df["tenure_days"] = (reference_date - df["signup_date"]).dt.days

In [25]:
#tracks who has been inactive for the last 30 days
df["is_inactive"] = (df["days_since_last_watch"] > 30).astype(int)

In [26]:
#billing history
df["high_risk_billing"] = (
    (df["price_increase_last_6m"] == True) & (df["payment_failures"] > 0)).astype(int)

In [27]:
#tracks if customer had more than 3 support tickets
df["high_support"] = (df["tickets_last_30d"] > 3).astype(int)

In [28]:
# Billing cycle as binary feature (yearly = 1, monthly = 0)
df["is_yearly"] = (df["billing_cycle"] == "yearly").astype(int)

In [29]:
# Effective price paid per month (yearly subscribers pay discounted rate)
df["effective_monthly_price"] = np.where(
    df["billing_cycle"] == "yearly",
    df["yearly_price"] / 12,
    df["monthly_price"]
)

In [ ]:
#has the customer cancelled?
df["is_cancelled"] = df["cancelled"].astype(int)

In [31]:
# Days until next renewal (closer to 0 = renewal imminent)
df["days_until_renewal"] = (df["next_renewal_date"] - reference_date).dt.days.clip(lower=0)

In [32]:
#converts string into numbers, also dropping the string columns
df = pd.get_dummies(df, columns=["country", "plan"], drop_first=False)

In [ ]:
#changing boolean types into int
df["price_increase_last_6m"] = df["price_increase_last_6m"].astype(int)

In [34]:
df = df.drop(columns=[
    "user_id", "signup_date", "start_date",
    "billing_cycle", "yearly_price", "cancelled",
    "cancellation_date", "last_renewal_date", "next_renewal_date"
], errors="ignore")

In [35]:
# sanity check - confirm no nulls slipped through
print("Null check after feature engineering:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("No nulls found" if df.isnull().sum().sum() == 0 else "")

Null check after feature engineering:
Series([], dtype: int64)
No nulls found


In [36]:
print("Shape after feature engineering:", df.shape)
print("\nNew features added:")
new_features = ["tenure_days", "is_inactive", "high_risk_billing", "high_support", "is_yearly", "effective_monthly_price", "is_cancelled", "days_until_renewal"]
print(df[new_features].describe().round(2))
print("\nFinal columns:")
print(list(df.columns))

Shape after feature engineering: (10000, 35)

New features added:
       tenure_days  is_inactive  high_risk_billing  high_support  is_yearly  \
count     10000.00     10000.00           10000.00      10000.00    10000.0   
mean        746.79         0.49               0.14          0.35        0.5   
std         434.02         0.50               0.34          0.48        0.5   
min           0.00         0.00               0.00          0.00        0.0   
25%         374.00         0.00               0.00          0.00        0.0   
50%         740.00         0.00               0.00          0.00        0.0   
75%        1120.00         1.00               0.00          1.00        1.0   
max        1500.00         1.00               1.00          1.00        1.0   

       effective_monthly_price  is_cancelled  days_until_renewal  
count                 10000.00      10000.00            10000.00  
mean                     14.28          0.15               99.90  
std                  

In [37]:
df.head()

,age,monthly_price,avg_watch_hours_per_week,number_of_logins_per_week,days_since_last_watch,completion_rate,tickets_last_30d,price_increase_last_6m,payment_failures,churn_type,...,country_Portugal,country_South Africa,country_South Korea,country_Spain,country_Thailand,country_UK,country_USA,plan_Basic,plan_Premium,plan_Standard
0,34,14.99,0.38,1,10,0.40,5,0,0,0,...,False,False,False,False,False,True,False,False,False,True
1,41,19.99,6.87,4,49,0.89,3,1,2,2,...,False,True,False,False,False,False,False,False,True,False
2,60,9.99,19.15,13,4,0.30,4,0,2,0,...,False,False,False,False,False,True,False,True,False,False
3,38,9.99,19.54,20,4,0.21,2,0,1,0,...,False,False,False,False,True,False,False,True,False,False
4,39,14.99,1.50,20,9,0.28,0,0,1,0,...,False,False,False,False,False,False,False,False,False,True


In [38]:
df.to_parquet("data/features.parquet", index=False)